In [44]:
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df = pd.read_csv(r'C:\Users\sinan\OneDrive\Desktop\5TH-sem_ojt\project1_diabetes\data\processed\sanin_p1w2_features.csv')


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_catogory","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [45]:
candidates = {
    "LogReg": ("scaled", LogisticRegression(max_iter=1000)),
    "Decision Tree": ("raw", DecisionTreeClassifier(max_depth=3, random_state=42)),
    "KNN": ("scaled", KNeighborsClassifier(n_neighbors=25)),
    "SVC balanced": ("scaled", SVC(class_weight="balanced", random_state=42)),
    "RF balanced": ("raw", RandomForestClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    )),
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  

In [46]:
print("rows:", len(df), "| train:", len(x_train), "| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("features after get_dummies:", x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
features after get_dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


## what is clone 
In Machine Learning, a clone means creating a new copy of a model with the same settings or parameters.

## re-split the data and watch the score move 


In [47]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y, test_size=0.2,
        random_state=rs, stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  # fresh copy

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))
        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
        row [name]= round(recall_score(yb,pred),3)
    rows.append(row)


splits = pd.DataFrame(rows).set_index("split")
print(splits)
spread = (splits.max() - splits.min()).round(3)
print("\nspread (best - worst ):\n",spread.sort_values(ascending=False))     

       LogReg  Decision Tree    KNN  SVC balanced  RF balanced
split                                                         
0       0.218          0.091  0.145         0.600        0.618
1       0.255          0.382  0.164         0.745        0.600
7       0.255          0.182  0.200         0.636        0.582
42      0.291          0.291  0.200         0.618        0.673
99      0.273          0.236  0.200         0.709        0.655

spread (best - worst ):
 Decision Tree    0.291
SVC balanced     0.145
RF balanced      0.091
LogReg           0.073
KNN              0.055
dtype: float64


In [48]:
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i, (tr,te) in enumerate(cv.split(x_train,y_train),1):
    print(f"fold {i}: train {len(tr)} score{len(te)}"
          f"diabetic in score fol {y_train.iloc[te].sum()}")

def make_model(kind, model):
    """scaled models get the scaler INSIDE a Pipeline, so it refits per fold."""
    if kind == "scaled":
        return Pipeline([("scale", StandardScaler()),("model",model)])
    return model
print()
for name, (kind, model) in candidates.items():
    print(f"{name:15s} -> {type(make_model(kind, model)).__name__}")

fold 1: train 608 score152diabetic in score fol 43
fold 2: train 608 score152diabetic in score fol 43
fold 3: train 608 score152diabetic in score fol 44
fold 4: train 608 score152diabetic in score fol 44
fold 5: train 608 score152diabetic in score fol 44

LogReg          -> Pipeline
Decision Tree   -> DecisionTreeClassifier
KNN             -> Pipeline
SVC balanced    -> Pipeline
RF balanced     -> RandomForestClassifier


## CROSS VALIDATE EVERY CANDIDATE

In [49]:
from sklearn.model_selection import cross_validate

rows = []
for name, (kind, model) in candidates.items():
  est = make_model(kind, model)
  res = cross_validate(
      est,
      x_train,
      y_train,
      cv=cv,
      scoring=["recall", "f1"],
  )
  single = clone(est).fit(x_train, y_train)
  rows.append({
      "model": name,
      "cv_recall": round(res["test_recall"].mean(), 3),
      "cv_std": round(res["test_recall"].std(), 3),
      "cv_f1": round(res["test_f1"].mean(), 3),
      "week6":round(recall_score(y_test,single.predict(x_test)),3),
      "folds":np.round(res["test_recall"],3).tolist(),
  })

cv_table = pd.DataFrame(rows).sort_values("cv_recall",ascending=False)
print(cv_table.to_string(index=False))  


        model  cv_recall  cv_std  cv_f1  week6                               folds
 SVC balanced      0.716   0.025  0.559  0.618 [0.744, 0.744, 0.682, 0.705, 0.705]
  RF balanced      0.661   0.031  0.539  0.673 [0.721, 0.651, 0.659, 0.636, 0.636]
       LogReg      0.303   0.073  0.389  0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296  0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.184   0.031  0.267  0.200 [0.209, 0.233, 0.159, 0.159, 0.159]


In [50]:
cv_table["low"] = (
    cv_table["cv_recall"] - cv_table["cv_std"]
).round(3)

cv_table["high"] = (
    cv_table["cv_recall"] + cv_table["cv_std"]
).round(3)


cv_table["clears_target"] = (
    cv_table["cv_recall"] >= TARGET_RECALL
)

cv_table["clears_guard"] = (
    cv_table["cv_f1"] >= GUARD_F1
)

print(cv_table[["model","cv_recall","low","high","cv_f1",
                "clears_target","clears_guard"]].to_string(index=False))


        model  cv_recall   low  high  cv_f1  clears_target  clears_guard
 SVC balanced      0.716 0.691 0.741  0.559           True          True
  RF balanced      0.661 0.630 0.692  0.539          False          True
       LogReg      0.303 0.230 0.376  0.389          False         False
Decision Tree      0.216 0.140 0.292  0.296          False         False
          KNN      0.184 0.153 0.215  0.267          False         False


## HYPERPARAMETER TUNING 
**Hyperparameter tuning is the process of finding the best settings for a Machine Learning model to improve its performance and accuracy**
1. What is GridSearchCV?

 GridSearchCV is a hyperparameter tuning technique that tests all specified combinations of hyperparameter values to find the best settings for a Machine Learning model.
 
 2. What is RandomizedSearchCV?

 RandomizedSearchCV is a hyperparameter tuning technique that tests a limited number of randomly selected combinations from the specified hyperparameter options or distributions.
 

## difference between hyper prameter and parameter
1. What is a Parameter?
Parameters are values that a Machine Learning model learns automatically from the training data.

Examples:
- Weights
- Bias
Example: In Linear Regression, the model learns the slope (m) and intercept (c) from the training data.

2. What is a Hyperparameter?
Hyperparameters are settings chosen by the user or adjusted through tuning to control how a Machine Learning model learns.

Examples:
- Learning rate
- Number of neighbors (n_neighbors)
- Maximum depth (max_depth)
- Number of trees (n_estimators)
Example: In a KNN model, we set n_neighbors = 5 to use the five nearest neighbors for prediction.

In [51]:
rf = RandomForestClassifier(random_state=42)
for setting, value in rf.get_params().items():
    print(f"{setting:25s} {value}")

bootstrap                 True
ccp_alpha                 0.0
class_weight              None
criterion                 gini
max_depth                 None
max_features              sqrt
max_leaf_nodes            None
max_samples               None
min_impurity_decrease     0.0
min_samples_leaf          1
min_samples_split         2
min_weight_fraction_leaf  0.0
monotonic_cst             None
n_estimators              100
n_jobs                    None
oob_score                 False
random_state              42
verbose                   0
warm_start                False


## build the grid and count the cost

In [52]:
param_grid ={
    "max_depth"       : [ 3,5,8,None],
    "min_samples_leaf" : [1,5,10],
    "class_weight"   : ["balanced","balanced_subsample"]
    }

combos = 1

for values in param_grid.values():
    combos *= len(values)

folds = cv.get_n_splits()
total_fits = combos*folds

print("combination:",combos)
print("cross-validation folds:",folds)
print("total model fits :", total_fits)

combination: 24
cross-validation folds: 5
total model fits : 120


## RUN THE GRID SEARCH ON THE LEADING FOREST

In [53]:
from sklearn.model_selection import GridSearchCV

gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring="recall",
    n_jobs=-1)

gs_recall.fit(x_train ,y_train)

print("Best settings:",gs_recall.best_params_)
print("Best CV recall :",round(gs_recall.best_score_,3))

top = pd.DataFrame(gs_recall.cv_results_)

print(
    top[["params","mean_test_score","std_test_score","rank_test_score"]]
    .sort_values("rank_test_score")
    .head()
    .to_string(index=False)
)

Best settings: {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}
Best CV recall : 0.707
                                                              params  mean_test_score  std_test_score  rank_test_score
{'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.706554        0.035347                1
 {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.692812        0.029575                2
 {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.688161        0.033213                3
 {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 5}         0.688055        0.023213                4
{'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10}         0.679281        0.040479                5
